# 3. Degeneracy and first loss of discrimination (Section 4, Table 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/main/notebooks/03_degeneracy.ipynb)

Enumerates all trees and counts distinct index values at 11 significant digits, as in the paper. Takes a few minutes.

Every check prints the value in the paper next to the recomputed value. The last cell fails if any of them differ.

In [1]:
# Setup: runs inside the repository, or on Google Colab (clones the repository first).
import os, sys, subprocess
if os.path.exists("../verify.py"):
    os.chdir("..")
elif not os.path.exists("verify.py"):
    if not os.path.exists("LOYOLA-PAPER"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Singati2/LOYOLA-PAPER.git"], check=True)
    os.chdir("LOYOLA-PAPER")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "scikit-learn"], check=True)
ROOT = os.getcwd()
for p in (ROOT, os.path.join(ROOT, "external"), os.path.join(ROOT, "external", "bp"), os.path.join(ROOT, "structural")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("working directory:", ROOT)

working directory: /Users/ganeshshiwakoti/LOYOLA-PAPER


In [2]:
import re, csv, time
import numpy as np, pandas as pd
RESULTS = []

def check(what, paper, got, tol=0.0):
    """Record one comparison between a value printed in the paper and the recomputed value."""
    if isinstance(paper, (int, float)) and not isinstance(paper, bool):
        ok = abs(float(paper) - float(got)) <= tol + 1e-12
    else:
        ok = paper == got
    RESULTS.append({"check": what, "paper": paper, "recomputed": got, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'} {what}: paper {paper}, recomputed {got}")
    return ok

def run(cmd, tail=25):
    """Run a repository script, show the end of its output and require exit status 0."""
    t = time.time()
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter's inline backend must not leak into scripts
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-tail:]))
    print(f"[exit {r.returncode}, {time.time() - t:.0f} s]")
    RESULTS.append({"check": f"script: {cmd}", "paper": "exit 0", "recomputed": f"exit {r.returncode}",
                    "status": "OK" if r.returncode == 0 else "MISMATCH"})
    return r.stdout

In [3]:
import networkx as nx
from collections import Counter
import octane_data as od
def profiles(n, chem=False):
    out = set()
    for T in nx.nonisomorphic_trees(n):
        d = dict(T.degree())
        if chem and max(d.values()) > 4:
            continue
        out.add(tuple(sorted(Counter(tuple(sorted((d[u], d[v]))) for u, v in T.edges()).items())))
    return out
def distinct(profs, a, b, g):
    return len({f"{od.lo_pairs([p for p, c in P for _ in range(c)], a, b, g):.10e}" for P in profs})

### The 106 trees of order 10 (Section 4.2)

In [4]:
trees10 = list(nx.nonisomorphic_trees(10)); P10 = profiles(10)
def values10(a, b, g):
    vals = set()
    for T in trees10:
        d = dict(T.degree())
        vals.add(f"{od.lo_pairs([tuple(sorted((d[u], d[v]))) for u, v in T.edges()], a, b, g):.10e}")
    return len(vals)
check("trees of order 10", 106, len(trees10))
check("distinct profiles", 79, len(P10))
check("LO(0,0,1) degeneracy (%)", 25.5, round(100 * (1 - values10(0, 0, 1) / 106), 1))
check("M1 degeneracy (%)", 83.0, round(100 * (1 - values10(0, 1, 0) / 106), 1))
check("M2 degeneracy (%)", 67.0, round(100 * (1 - values10(1, 0, 0) / 106), 1))
check("chi distinct values", 77, values10(0, -0.5, 0))

OK       trees of order 10: paper 106, recomputed 106
OK       distinct profiles: paper 79, recomputed 79
OK       LO(0,0,1) degeneracy (%): paper 25.5, recomputed 25.5
OK       M1 degeneracy (%): paper 83.0, recomputed 83.0
OK       M2 degeneracy (%): paper 67.0, recomputed 67.0
OK       chi distinct values: paper 77, recomputed 77


True

### Table 2: first order with fewer values than profiles

In [5]:
rows = {}
for n in range(12, 18):
    for chem in (False, True):
        if not chem and n == 17:
            continue
        P = profiles(n, chem)
        rows[(n, chem)] = (len(P), distinct(P, 0, 0, 1), distinct(P, -0.5, 0, 0))
        print(n, "max degree 4" if chem else "all trees", rows[(n, chem)])
check("LO(0,0,1), all trees: order 13 values", 566, rows[(13, False)][1])
check("LO(0,0,1), all trees: order 13 profiles", 570, rows[(13, False)][0])
check("LO(0,0,1), all trees: order 12 at the floor", True, rows[(12, False)][1] == rows[(12, False)][0])
check("LO(0,0,1), max degree 4: order 16 values", 809, rows[(16, True)][1])
check("LO(0,0,1), max degree 4: order 16 profiles", 810, rows[(16, True)][0])
check("R, all trees: order 14 values", 1095, rows[(14, False)][2])
check("R, all trees: order 14 profiles", 1100, rows[(14, False)][0])
check("R, max degree 4: order 17 values", 1191, rows[(17, True)][2])
check("R, max degree 4: order 17 profiles", 1194, rows[(17, True)][0])

12 all trees (294, 294, 294)
12 max degree 4 (137, 137, 137)
13 all trees (570, 566, 570)
13 max degree 4 (223, 223, 223)
14 all trees (1100, 1090, 1095)
14 max degree 4 (350, 350, 350)


15 all trees (2119, 2093, 2100)
15 max degree 4 (538, 538, 538)


16 all trees (4069, 3996, 4007)


16 max degree 4 (810, 809, 810)


17 max degree 4 (1194, 1189, 1191)
OK       LO(0,0,1), all trees: order 13 values: paper 566, recomputed 566
OK       LO(0,0,1), all trees: order 13 profiles: paper 570, recomputed 570
OK       LO(0,0,1), all trees: order 12 at the floor: paper True, recomputed True
OK       LO(0,0,1), max degree 4: order 16 values: paper 809, recomputed 809
OK       LO(0,0,1), max degree 4: order 16 profiles: paper 810, recomputed 810
OK       R, all trees: order 14 values: paper 1095, recomputed 1095
OK       R, all trees: order 14 profiles: paper 1100, recomputed 1100
OK       R, max degree 4: order 17 values: paper 1191, recomputed 1191
OK       R, max degree 4: order 17 profiles: paper 1194, recomputed 1194


True

### Section 4.2: what gamma recovers off the ratio plane (all trees of order 16)

In [6]:
P16 = profiles(16)
check("profiles of order 16", 4069, len(P16))
for a, b, g, want in ((0, 0.5, 0, 3475), (0, 1/3, 0, 3476), (0, 0.5, 1, 4069), (0, 1/3, 1, 4069),
                      (0.5, 0, 0, 3940), (0.5, 0, 1, 4067), (-0.5, 0, 0, 4007), (-0.5, 0, 1, 4068), (0.5, -1, 0, 3996)):
    check(f"distinct values of LO({a:g},{b:.3g},{g:g})", want, distinct(P16, a, b, g))

OK       profiles of order 16: paper 4069, recomputed 4069
OK       distinct values of LO(0,0.5,0): paper 3475, recomputed 3475
OK       distinct values of LO(0,0.333,0): paper 3476, recomputed 3476
OK       distinct values of LO(0,0.5,1): paper 4069, recomputed 4069
OK       distinct values of LO(0,0.333,1): paper 4069, recomputed 4069
OK       distinct values of LO(0.5,0,0): paper 3940, recomputed 3940
OK       distinct values of LO(0.5,0,1): paper 4067, recomputed 4067
OK       distinct values of LO(-0.5,0,0): paper 4007, recomputed 4007
OK       distinct values of LO(-0.5,0,1): paper 4068, recomputed 4068
OK       distinct values of LO(0.5,-1,0): paper 3996, recomputed 3996


## Summary

In [7]:
summary = pd.DataFrame(RESULTS)
display(summary) if "display" in dir() else print(summary.to_string())
n_bad = int((summary.status != "OK").sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
assert n_bad == 0, "some recomputed values differ from the paper"

                                          check paper recomputed status
0                             trees of order 10   106        106     OK
1                             distinct profiles    79         79     OK
2                      LO(0,0,1) degeneracy (%)  25.5       25.5     OK
3                             M1 degeneracy (%)  83.0       83.0     OK
4                             M2 degeneracy (%)  67.0       67.0     OK
5                           chi distinct values    77         77     OK
6         LO(0,0,1), all trees: order 13 values   566        566     OK
7       LO(0,0,1), all trees: order 13 profiles   570        570     OK
8   LO(0,0,1), all trees: order 12 at the floor  True       True     OK
9      LO(0,0,1), max degree 4: order 16 values   809        809     OK
10   LO(0,0,1), max degree 4: order 16 profiles   810        810     OK
11                R, all trees: order 14 values  1095       1095     OK
12              R, all trees: order 14 profiles  1100       1100